# Alternative ELFI configurations

Alternative ELFI model: nodes separately for SIR and BSI

In [2]:
# Testing aggregates

elfi.new_model()

bs = 10

SIR_obs = prop_to_nSIR(propSIR_simulator(0.734, 1/30, nt = n_weeks, N = pop_size, batch_size = 1), pop_size)
OR_hat = get_OR_hat(or_data, clade = "A", dataset = "NORM")
bsi_obs = sum_over_bsi(col_to_BSI(SIR_obs, OR_hat = OR_hat, is_prop = False))#.flatten()
print(bsi_obs.shape)

obsS1 = BSI_max_t(bsi_obs)
obsS2 = BSI_max(bsi_obs)
obs_summaries = (obsS1, obsS2)

mu_OR, sd_OR = get_OR_hat_pars(or_data, clade = "A", dataset = "NORM")

bsi_parameters = {"or_data": or_data, "clade":"A", "dataset":"NORM", "theta_c":1, "theta_bsi":0.3} # todo: deprecate this

# Clancy et al: uninformative gamma priors for beta and gamma
beta = elfi.Prior(scipy.stats.gamma,1,0, 0.1)
gamma = elfi.Prior(scipy.stats.gamma,1,0, 0.1)

nt = elfi.Constant(n_weeks)
N = elfi.Constant(pop_size)

mu_OR = elfi.Constant(mu_OR)
sd_OR = elfi.Constant(sd_OR)
OR_hat = elfi.RandomVariable(scipy.stats.norm, mu_OR, sd_OR)

print(np.array([obsS1, obsS2]).reshape(-1,1).transpose().shape)
SIR = elfi.Simulator(propSIR_simulator, beta, gamma, nt, N, observed = bsi_obs) # techincally, observations do not match this node.
nSIR = elfi.Operation(prop_to_nSIR, SIR, N)

is_prop = elfi.Constant(False)

theta_c = elfi.Constant(1)
theta_bsi = elfi.Constant(0.28)
BSI = elfi.Operation(col_to_BSI, nSIR, OR_hat, theta_c, theta_bsi, is_prop)
sumBSI = elfi.Operation(sum_over_bsi, BSI) # is there any way of inputting observed data to this node?


S1 = elfi.Summary(BSI_max_t, sumBSI)
S2 = elfi.Summary(BSI_max, sumBSI)


#d = elfi.Discrepancy(euclidean_distance, S1, S2, obs_summaries)

d = elfi.Distance('euclidean', S1, S2)

elfi.draw(d)

NameError: name 'elfi' is not defined